For casme dataset, load the data per person from ftp server, one after other in a thread in background, it has frames stored, get only the color values, no depth needed, store them to ./data/casme/person_id/, this will be fed to the motion amplification.

## CASME3 Data Loader (FTP)

This script connects to the CASME3 FTP server, searches for subject directories containing "color" folders, and downloads the frames to `./data/casme/{subject_id}/`. 
It runs in a background thread so you can continue working while data downloads.

**Instructions:**
1. Update `FTP_HOST`, `FTP_USER`, and `FTP_PASS` with your credentials.
2. Run the cell below.
3. The download will happen in the background. Check the `./data/casme` folder for progress.

In [ ]:
import queue
import threading
import time
import ftplib
import os
import shutil
import zipfile
import pandas as pd
import re
from dotenv import load_dotenv

STOP_SIGNAL = "STOP"
load_dotenv()


def on_rm_error(func, path, exc_info):
    """
    Error handler for shutil.rmtree.
    If the error is due to an access error (read only file)
    it attempts to add write permission and then retries.
    If the error is due to the file being used by another process,
    it waits a bit and retries.
    """
    import stat
    # Is the error an access error?
    if not os.access(path, os.W_OK):
        os.chmod(path, stat.S_IWRITE)
        try:
            func(path)
            return
        except Exception:
            pass
    
    # Maybe locked?
    print(f"Warning: Could not delete {path}. Retrying in 1s...")
    time.sleep(1)
    try:
        # Try to chmod again just in case
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"Failed to force delete {path}: {e}")

def robust_rmtree(path, retries=5, delay=1.0):
    if not os.path.exists(path):
        return
    
    # Ensure the directory itself is writable before we start
    try:
        import stat
        os.chmod(path, stat.S_IWRITE)
    except:
        pass

    for i in range(retries):
        try:
            # Check if any files are somehow still open/locked? 
            # We can't easily check for open handles in pure python without psutil
            # but usually GC takes care of it.
            # Explicit gc might help if object references are holding files open.
            import gc
            gc.collect()
            
            shutil.rmtree(path, onerror=on_rm_error)
            print(f"Successfully deleted {path}")
            return
        except OSError as e:
            if i < retries - 1:
                time.sleep(delay)
            else:
                print(f"Warning: Failed to delete {path} after {retries} attempts: {e}")
                # Fallback: Try to use system command on Windows
                if os.name == 'nt':
                    try:
                        print("Attempting system shell delete...")
                        os.system(f'rmdir /S /Q "{path}"')
                    except Exception as sys_e:
                        print(f"System shell delete failed: {sys_e}")


def download_casme3_generator(host, user, password, local_root_dir, excel_path, parts_config, stop_event):
    """
    Iterates through dataset parts (A, B... per config), downloads annotated subject zips, 
    extracts them, and yields the local path.
    """
    print(f"Connecting to FTP {host}...")
    ftp = None
    try:
        ftp = ftplib.FTP(host)
        ftp.login(user, password)
    except Exception as e:
        print(f"FTP Connection Failed: {e}")
        return

    # 1. Load Annotations
    valid_subjects = set()
    try:
        if excel_path and os.path.exists(excel_path):
            df = pd.read_excel(excel_path)
            # Heuristic to find subject column
            valid_col = None
            for col in df.columns:
                if 'sub' in str(col).lower():
                    valid_col = col
                    break
            
            if valid_col:
                # robust ID extraction
                raw_list = df[valid_col].astype(str).tolist()
                for item in raw_list:
                    match = re.search(r'(\d+)', item)
                    if match:
                        num_str = match.group(1)
                        valid_subjects.add(num_str) 
                        valid_subjects.add(str(int(num_str)))
                
                print(f"Loaded {len(valid_subjects)} valid subject IDs from annotations.")
            else:
                print(f"Warning: Could not find 'Subject' column in {excel_path}. Processing all.")
        else:
            print(f"Annotation file not found at {excel_path}. Processing all subjects found on FTP.")
    except Exception as e:
        print(f"Error reading Excel: {e}")

    try:
        # 2. Iterate Configured Parts
        for part_name, relative_path in parts_config.items():
            if stop_event.is_set(): break

            print(f"Searching {part_name} at {relative_path}...")
            
            file_list = []
            try:
                # List files in the target directory
                ftp.retrlines(f'NLST {relative_path}', file_list.append)
                print(f"Found {len(file_list)} files in {part_name}")
            except ftplib.error_perm as e:
                print(f"Skipping {part_name} (Not found or No Access): {e}")
                continue

            file_list.sort() 
            
            processed_count = 0
            
            for zip_filename in file_list:
                if stop_event.is_set(): 
                    print("Stop signal received. Aborting download loop.")
                    return
                
                # Check extension (case insensitive)
                if not zip_filename.lower().endswith('.zip'):
                    continue
                
                # Extract numeric ID from filename
                base_name = os.path.basename(zip_filename.replace('\\', '/'))
                match = re.search(r'(\d+)', base_name)
                
                if not match:
                    continue
                
                subject_id_raw = match.group(1)
                subject_int = str(int(subject_id_raw)) # "01" -> "1"
                
                # Filter Logic
                is_valid = (not valid_subjects) or \
                           (subject_id_raw in valid_subjects) or \
                           (subject_int in valid_subjects)

                if valid_subjects and not is_valid:
                    continue

                # Prepare Local Paths
                # Unique folder name: sub{ID}_{Part}
                safe_id = f"sub{subject_id_raw}_{part_name}"
                extract_dir = os.path.join(local_root_dir, safe_id)
                local_zip_path = os.path.join(local_root_dir, f"{safe_id}.zip")

                # Check if cached
                if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 0:
                    print(f"Found cached data for {safe_id}. Yielding...")
                    yield extract_dir
                    continue
                
                # Clean partial previous attempts
                robust_rmtree(extract_dir)
                os.makedirs(extract_dir, exist_ok=True)

                print(f"Downloading {zip_filename}...")
                
                # Create directory path logic
                if '/' in zip_filename or '\\' in zip_filename:
                     remote_file = zip_filename.replace('\\', '/')
                else:
                     remote_file = f"{relative_path}/{zip_filename}"
                
                remote_file = remote_file.replace('//', '/')

                # Download ZIP
                try:
                    with open(local_zip_path, 'wb') as f:
                        ftp.retrbinary(f"RETR {remote_file}", f.write)
                    
                    # Verify file size
                    if os.path.getsize(local_zip_path) < 100:
                        print(f"Warning: Downloaded file {local_zip_path} is too small. Deleting.")
                        try: os.remove(local_zip_path)
                        except: pass
                        continue

                except Exception as e:
                    print(f"Download failed for {zip_filename} (Path: {remote_file}): {e}")
                    try: os.remove(local_zip_path)
                    except: pass
                    robust_rmtree(extract_dir)
                    continue

                if stop_event.is_set():
                    try: os.remove(local_zip_path)
                    except: pass
                    return

                # Extract ZIP
                print(f"Extracting {safe_id}...")
                try:
                    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
                        # Ensure we don't hold the file open implicitly
                        zip_ref.extractall(extract_dir)
                    
                    if len(os.listdir(extract_dir)) == 0:
                         print(f"Warning: {zip_filename} extraction resulted in empty dir.")
                    else:
                         yield extract_dir
                         processed_count += 1
                         
                except zipfile.BadZipFile:
                    print(f"Corrupt Zip File: {zip_filename}")
                except Exception as e:
                    print(f"Extraction Error: {e}")
                finally:
                    # Explicitly remove zip file
                    try:
                        if os.path.exists(local_zip_path):
                            os.remove(local_zip_path)
                    except Exception as e:
                        print(f"Warning: Could not remove zip {local_zip_path}: {e}")
            
            if processed_count == 0:
                print(f"Warning: No valid zip files found or downloaded in {part_name}")

    except Exception as e:
        print(f"Generator Error: {e}")
    finally:
        if ftp:
            try: ftp.quit() 
            except: pass


def _producer_thread(host, user, password, local_root, excel_path, config, q, stop_event):
    """
    Driven by the generator. Puts resulting paths into the provided queue.
    """
    try:
        gen = download_casme3_generator(host, user, password, local_root, excel_path, config, stop_event)
        for path in gen:
            if stop_event.is_set(): break
            while not stop_event.is_set():
                try:
                    q.put(path, timeout=1)
                    break 
                except queue.Full:
                    continue
    except Exception as e:
        print(f"Producer Thread crashed: {e}")
    finally:
        # Use put_nowait or timeout to prevent hanging if queue is full and consumer is gone
        try:
            q.put(STOP_SIGNAL, timeout=5)
        except queue.Full:
            print("Warning: Could not send STOP signal (Queue full).")
        print("Background download process finished.")


class CASMEDataLoader:
    def __init__(self, excel_path, local_root="./data/casme", buffer_size=1):
        # Load from .env
        self.host = os.getenv("FTP_HOST")
        self.user = os.getenv("FTP_USER")
        self.password = os.getenv("FTP_PASS")
        
        if not all([self.host, self.user, self.password]):
            raise ValueError("Missing FTP credentials in .env file (FTP_HOST, FTP_USER, FTP_PASS)")
            
        self.excel_path = excel_path
        self.local_root = local_root
        
        # Configure FTP paths per dataset structure (update if needed)
        self.parts_config = {
            "Part_A": "part_A/data/Compressed_version1_seperate_compress",
            "Part_B": "part_B/Compressed_version1_seperate_compress", 
        }
        
        # Queue maxsize limits disk usage.
        self.data_queue = queue.Queue(maxsize=buffer_size) 
        self._thread = None
        self._stop_event = threading.Event()
        
    def start(self):
        """Starts the background downloading thread."""
        if self._thread and self._thread.is_alive():
            print("Loader already running.")
            return
            
        self._stop_event.clear()
        self._thread = threading.Thread(
            target=_producer_thread,
            args=(self.host, self.user, self.password, self.local_root, self.excel_path, self.parts_config, self.data_queue, self._stop_event)
        )
        self._thread.daemon = True
        self._thread.start()
        print("Background data loading started...")

    def stop(self):
        """sends stop signal to background thread."""
        print("Stopping background loader...")
        self._stop_event.set()
        
        # Drain the queue
        try:
            while not self.data_queue.empty():
                self.data_queue.get_nowait()
        except:
            pass

    def get_next_subject(self):
        """
        Returns (subject_path) for next subject.
        Blocks if waiting for download.
        Returns None if exhausted.
        """
        while True:
            try:
                item = self.data_queue.get(block=True, timeout=1) 
                
                if item == STOP_SIGNAL:
                    self.data_queue.put(STOP_SIGNAL) 
                    return None
                    
                return item
                
            except queue.Empty:
                if self._stop_event.is_set():
                    return None
                    
                if self._thread and not self._thread.is_alive():
                    return None
                continue 
    
    def cleanup_subject(self, subject_path):
        """
        Deletes the local data for a subject with retry logic.
        """
        robust_rmtree(subject_path)


In [7]:
# Usage Example
excel_file = "casme/CASME3_part_A_1.xls" # Path to your annotation file

# Initialized from .env
loader = CASMEDataLoader(excel_file) # No longer need to pass credentials explicitly

try:
    loader.start()
    
    # Process subjects as they arrive
    count = 0
    while True:
        print("\nWaiting for next subject...")
        subject_path = loader.get_next_subject()
        
        if subject_path is None:
            print("No more subjects.")
            break
            
        print(f"Ready for training: {subject_path}")
        
        # Simulate training time
        time.sleep(2) 
        
        # Cleanup when done with this subject
        loader.cleanup_subject(subject_path)
        
        count += 1
        # Stop after 2 subjects to confirm it works without downloading everything
        if count >= 1:
            print("Test run complete (processed 2 subjects). Stopping loader.")
            break

except KeyboardInterrupt:
    print("\nUser interrupted training.")
finally:
    loader.stop()
    print("All downloads finished or stopped.")


Connecting to FTP 74.220.215.205...Background data loading started...

Waiting for next subject...

Annotation file not found at casme/CASME3_part_A_1.xls. Processing all subjects found on FTP.
Searching Part_A at part_A/data/Compressed_version1_seperate_compress...
Found 102 files in Part_A

User interrupted training.
Stopping background loader...
All downloads finished or stopped.
